# SIDER 

In [ ]:
import csv
import pandas as pd

input_tsv = "../prototype_old_dont use/meddra_all_se.tsv"
output_csv = "../prototype_old_dont use/meddra_all_se.csv" 

df = pd.read_csv(input_tsv, sep="\t", low_memory=False)

print("Columns:")
print(df.columns)

print("\nFirst 5 rows:")
print(df.head())
df.to_csv(output_csv, index=False)


/var/folders/0x/bpcwfpyd38x9ypvxs83s6drw0000gn/T/ipykernel_17696/3412866894.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Columns:
Index(['CID100000085', 'CID000010917', 'C0000729', 'LLT', 'C0000729.1',
       'Abdominal cramps'],
      dtype='object')

First 5 rows:
   CID100000085  CID000010917  C0000729  LLT C0000729.1       Abdominal cramps
0  CID100000085  CID000010917  C0000729   PT   C0000737         Abdominal pain
1  CID100000085  CID000010917  C0000737  LLT   C0000737         Abdominal pain
2  CID100000085  CID000010917  C0000737   PT   C0687713  Gastrointestinal pain
3  CID100000085  CID000010917  C0000737   PT   C0000737         Abdominal pain
4  CID100000085  CID000010917  C0002418  LLT   C0002418              Amblyopia


In [2]:
import pandas as pd

# Paths
input_csv = "./meddra_all_se.csv"
output_csv = "./L_raw_sider_edges.csv"

# Read file with no header
df = pd.read_csv(
    input_csv,
    header=None,
    low_memory=False
)


# Keep only PT-level rows
df = df[df[3] == "PT"]

# Select required columns by index
# 0 -> STITCH compound ID -- drug ID
# 4 -> MedDRA PT ID -- adr ID
#5 ->  adr name
df = df[[0, 4, 5]]

# Rename columns for clarity
df.columns = ["drug_id", "adr_id", "adr_name"]

# Drop duplicate (drug_id, adr_id) pairs
df = df.drop_duplicates()

# Save raw SIDER edge list
df.to_csv(output_csv, index=False)

print(f"L_raw created with {len(df)} unique drug–ADR edges")


L_raw created with 145321 unique drug–ADR edges


In [3]:
df.head()

,drug_id,adr_id,adr_name
1,CID100000085,C0000737,Abdominal pain
3,CID100000085,C0687713,Gastrointestinal pain
6,CID100000085,C0002418,Amblyopia
8,CID100000085,C0002871,Anaemia
10,CID100000085,C0232462,Decreased appetite


In [ ]:
import re

input_csv = "./L_raw_sider_edges.csv"
output_csv = "./L_clean_sider_edges.csv"

df = pd.read_csv(input_csv)

def extract_pubchem_cid(drug_id):
    if isinstance(drug_id, str):
        #cid with CID12345 or with CIDm or CIDs
        match = re.match(r"CID(?:m|s)?(\d+)", drug_id)
        if match:
            return int(match.group(1))
    return None

# Normalize drug identifiers
df["drug_cid"] = df["drug_id"].apply(extract_pubchem_cid)
# Drop rows where conversion failed
df = df.dropna(subset=["drug_cid"])

# Ensure integer type
df["drug_cid"] = df["drug_cid"].astype(int)

# Drop duplicate edges (based on normalized identity)
df = df.drop_duplicates(subset=["drug_cid", "adr_id"])

# Save enriched cleaned list
df.to_csv(output_csv, index=False)


In [5]:
sider_edges = pd.read_csv("L_clean_sider_edges.csv")

drug_names = pd.read_csv(
    "drug_names.tsv",
    sep="\t",
    header=None,
    names=["drug_id", "drug_name"]
)

sider_named = sider_edges.merge(
    drug_names,
    on="drug_id",
    how="left"
)

# sider_named.head()


In [ ]:
SALT_WORDS = [
    "hydrochloride",
    "sodium",
    "potassium",
    "acetate",
    "sulfate",
    "phosphate"
]

salt_pattern = r"\b(" + "|".join(SALT_WORDS) + r")\b"

def normalize_sider_name(name):
    if pd.isna(name):
        return None
    
    # lowercase
    name = name.lower()
    
    # remove punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    
    # remove salt words
    name = re.sub(salt_pattern, " ", name)
    
    # collapse multiple spaces
    name = re.sub(r"\s+", " ", name)
    
    # trim
    return name.strip()

# Create normalized name column
sider_named["sider_norm_name"] = sider_named["drug_name"].apply(normalize_sider_name)

# sider_named.head()
sider_named[["drug_name", "sider_norm_name"]].drop_duplicates().head(20)

sider_named.to_csv("./L_clean_norm_named_sider.csv")


# LINCS: 

In [ ]:
# Input and output paths
input_txt = "./GSE92742_Broad_LINCS_inst_info.txt"
output_csv = "./GSE92742_Broad_LINCS_inst_info.csv"

# Read tab-delimited text file
df = pd.read_csv(
    input_txt,
    sep="\t",
    low_memory=False
)

# Write to CSV
df.to_csv(output_csv, index=False)

print("Shape:", df.shape)


Conversion complete
Shape: (1319138, 11)


inst_id : Unique identifier for a LINCS experiment instance. Each inst_id corresponds to one gene-expression profile produced under a specific condition (drug, dose, time, cell line).

pert_id: LINCS perturbation identifier. For drugs, this is usually a BRD-ID (e.g., BRD-A60070924). 
This is the primary compound identifier inside LINCS.

pert_iname: Human-readable perturbation name. Examples: alpha-estradiol, vorinostat, dexamethasone.  useful for sanity checks and name-based matching (but not canonical).

pert_type: Type of perturbation. Important values:
    trt_cp → small-molecule drug treatment -- our thing. 
    ctl_vehicle → control (e.g., DMSO) no drug. jst control group
    trt_sh → gene knockdown
    trt_oe → gene overexpression



In [46]:
import pandas as pd

# Load LINCS metadata
df_lincs = pd.read_csv(
    "GSE92742_Broad_LINCS_inst_info.csv",
    low_memory=False
)

# Keep only small-molecule drugs
df_lincs_drugs = df_lincs[df_lincs["pert_type"] == "trt_cp"]

# Keep unique LINCS compounds
lincs_compounds = df_lincs_drugs.drop_duplicates()

print("Number of LINCS drug perturbations:", len(lincs_compounds))

lincs_compounds.to_csv(
    "LINCS_drug_universe.csv",
    index=False
)

lincs_compounds.head()

Number of LINCS drug perturbations: 672128


,inst_id,rna_plate,rna_well,pert_id,pert_iname,pert_type,pert_dose,pert_dose_unit,pert_time,pert_time_unit,cell_id
12,ASG001_MCF7_24H_X1_B7_DUO52HI53LO:A06,ASG001_MCF7_24H_X1,A06,BRD-A60070924,alpha-estradiol,trt_cp,10.00,um,24,h,MCF7
13,ASG001_MCF7_24H_X1_B7_DUO52HI53LO:B06,ASG001_MCF7_24H_X1,B06,BRD-A60070924,alpha-estradiol,trt_cp,2.00,um,24,h,MCF7
14,ASG001_MCF7_24H_X1_B7_DUO52HI53LO:C06,ASG001_MCF7_24H_X1,C06,BRD-A60070924,alpha-estradiol,trt_cp,0.40,um,24,h,MCF7
15,ASG001_MCF7_24H_X1_B7_DUO52HI53LO:D06,ASG001_MCF7_24H_X1,D06,BRD-A60070924,alpha-estradiol,trt_cp,0.08,um,24,h,MCF7
16,ASG001_MCF7_24H_X1_B7_DUO52HI53LO:E06,ASG001_MCF7_24H_X1,E06,BRD-A60070924,alpha-estradiol,trt_cp,10.00,um,24,h,MCF7


### NORMALIZE LINCS pert_iname

In [54]:
lincs = pd.read_csv("LINCS_drug_universe.csv")

# Same salt words as SIDER
SALT_WORDS = [
    "hydrochloride",
    "sodium",
    "potassium",
    "acetate",
    "sulfate",
    "phosphate"
]

salt_pattern = r"\b(" + "|".join(SALT_WORDS) + r")\b"

def normalize_lincs_name(name):
    if pd.isna(name):
        return None
    
    # lowercase
    name = name.lower()
    
    # remove punctuation
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    
    # remove salt/formulation words
    name = re.sub(salt_pattern, " ", name)
    
    # collapse multiple spaces
    name = re.sub(r"\s+", " ", name)
    
    # trim
    return name.strip()

lincs["lincs_norm_name"] = lincs["pert_iname"].apply(normalize_lincs_name)
# lincs.head()
lincs.to_csv("./L_lincs_norm_name.csv")


In [56]:
sider = pd.read_csv("./L_clean_norm_named_sider.csv")
lincs= pd.read_csv("./L_lincs_norm_name.csv")

sider_names = set(
    sider["sider_norm_name"]
    .dropna()
    .unique()
)

lincs_names = set(
    lincs["lincs_norm_name"]
    .dropna()
    .unique()
)

common_names = sider_names & lincs_names

print("Unique SIDER drugs (normalized):", len(sider_names))
print("Unique LINCS drugs (normalized):", len(lincs_names))
print("Common drugs (name-based intersection):", len(common_names))


Unique SIDER drugs (normalized): 1340
Unique LINCS drugs (normalized): 19799
Common drugs (name-based intersection): 698


In [57]:
sider_common = sider[sider["sider_norm_name"].isin(common_names)]
lincs_common = lincs[lincs["lincs_norm_name"].isin(common_names)]

print("SIDER drugs:", sider_common["sider_norm_name"].nunique())
print("LINCS drugs:", lincs_common["lincs_norm_name"].nunique())


SIDER drugs: 698
LINCS drugs: 698


In [58]:
common_dataset = sider_common.merge(
    lincs_common[["pert_id", "pert_iname", "lincs_norm_name"]],
    left_on="sider_norm_name",
    right_on="lincs_norm_name",
    how="inner"
)

common_dataset = common_dataset[[
    "drug_id",          # STITCH ID
    "drug_cid",         # PubChem CID (non-canonical but useful)
    "drug_name",
    "sider_norm_name",  # shared key
    "pert_id",          # LINCS BRD ID
    "pert_iname",
    "adr_id"
]]
common_dataset = common_dataset.drop_duplicates()

common_dataset.to_csv( "SIDER_LINCS_common_drug_ADR.csv", index=False)


In [59]:
common_dataset.head()

,drug_id,drug_cid,drug_name,sider_norm_name,pert_id,pert_iname,adr_id
0,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0002170
11,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0232462
22,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009450
33,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009806
44,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0011603


# CTD:

## ctd chemicals:

In [94]:
import pandas as pd
cols = [
    "ChemicalName",
    "ChemicalID",
    "CasRN",
    "PubChemCID",
    "PubChemSID",
    "DTXSID",
    "InChIKey",
    "Definition",
    "ParentIDs",
    "TreeNumbers",
    "ParentTreeNumbers",
    "MESHSynonyms",
    "CTDCuratedSynonyms"
]

df = pd.read_csv(
    "CTD_chemicals.tsv.gz",   # must be original CTD file
    sep="\t",
    comment="#",
    header=None,
    names=cols,
    dtype=str,
    low_memory=False
)

# # Drop rows with empty ChemicalName
df = df[df["ChemicalName"].notna()]
df = df[df["ChemicalName"].str.strip() != ""]

df.to_csv("CTD_chemicals.csv")

df.head()
# print(df.columns)


,ChemicalName,ChemicalID,CasRN,PubChemCID,PubChemSID,DTXSID,InChIKey,Definition,ParentIDs,TreeNumbers,ParentTreeNumbers,MESHSynonyms,CTDCuratedSynonyms
0,(0.017ferrocene)amylose,MESH:C089250,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D000075163|MESH:D000688|MESH:D005296,D01.490.200/C089250|D02.691.657/C089250|D09.30...,D01.490.200|D02.691.657|D09.301.915.361|D09.69...,(0.017 ferrocene)amylose,NaN
1,001-C8-NBD,MESH:C114385,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842|MESH:D010069,D03.383.129.462.580/C114385|D12.644.456/C114385,D03.383.129.462.580|D12.644.456,001 C8 NBD|H-MeTyr-Arg-MeArg-D-Leu-NH(CH2)8NH-...,NaN
2,001-C8 oligopeptide,MESH:C114386,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842,D12.644.456/C114386,D12.644.456,001 C8 oligopeptide|H-MeTyr-Arg-MeArg-D-Leu-NH...,NaN
3,"0231A , Streptomyces",MESH:C434150,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434150,D03.633.400,NaN,NaN
4,"0231B, Streptomyces",MESH:C434149,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434149,D03.633.400,NaN,NaN


In [88]:
df = df.reset_index(drop=True)
df.to_csv("CTD_chemicals.csv", index=False)


In [89]:
df = pd.read_csv("CTD_chemicals.csv", low_memory=False)
df.head()


,ChemicalName,ChemicalID,CasRN,PubChemCID,PubChemSID,DSSToxSubstanceID,InChIKey,Definition,ParentIDs,TreeNumbers,ParentTreeNumbers,Synonyms
0,MESH:C089250,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D000075163|MESH:D000688|MESH:D005296,D01.490.200/C089250|D02.691.657/C089250|D09.30...,D01.490.200|D02.691.657|D09.301.915.361|D09.69...,(0.017 ferrocene)amylose,NaN
1,MESH:C114385,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842|MESH:D010069,D03.383.129.462.580/C114385|D12.644.456/C114385,D03.383.129.462.580|D12.644.456,001 C8 NBD|H-MeTyr-Arg-MeArg-D-Leu-NH(CH2)8NH-...,NaN
2,MESH:C114386,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842,D12.644.456/C114386,D12.644.456,001 C8 oligopeptide|H-MeTyr-Arg-MeArg-D-Leu-NH...,NaN
3,MESH:C434150,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434150,D03.633.400,NaN,NaN
4,MESH:C434149,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434149,D03.633.400,NaN,NaN


In [ ]:
import re

SALT_PATTERN = re.compile(
    r'\b(hydrochloride|sodium|potassium|acetate|sulfate|phosphate|'
    r'tartrate|succinate|fumarate)\b'
)

def normalize_drug_name(name: str) -> str:
    if pd.isna(name):
        return ""

    name = name.lower()
    name = re.sub(r'[^a-z0-9\s]', ' ', name)   # remove punctuation
    name = SALT_PATTERN.sub('', name)          # remove salts
    name = re.sub(r'\s+', ' ', name)           # collapse spaces
    return name.strip()

-
ctd = pd.read_csv("./CTD_chemicals.csv")

-
# Normalize names
-
ctd["ctd_norm_name"] = ctd["ChemicalName"].apply(normalize_drug_name)

-
# Sanity checks
-
print(ctd[["ChemicalID", "ChemicalName", "ctd_norm_name"]].head(10))

print("\nUnique normalized names:", ctd["ctd_norm_name"].nunique())
print("Total chemicals:", len(ctd))

-
# Save normalized file
-
ctd.to_csv("CTD_chemicals_normalized.csv", index=False)


        ChemicalID             ChemicalName            ctd_norm_name
0     MESH:C089250  (0.017ferrocene)amylose   0 017ferrocene amylose
1     MESH:C114385               001-C8-NBD               001 c8 nbd
2     MESH:C114386      001-C8 oligopeptide      001 c8 oligopeptide
3     MESH:C434150     0231A , Streptomyces       0231a streptomyces
4     MESH:C434149      0231B, Streptomyces       0231b streptomyces
5  MESH:C000620092          027075 compound          027075 compound
6     MESH:C533344                  0433YC1                  0433yc1
7     MESH:C533345                  0433YC2                  0433yc2
8     MESH:C046983  06-Paris-LA-66 protocol  06 paris la 66 protocol
9     MESH:C585814         071031B compound         071031b compound

Unique normalized names: 179010
Total chemicals: 179408


In [96]:
ctd.head()

,Unnamed: 0,ChemicalName,ChemicalID,CasRN,PubChemCID,PubChemSID,DTXSID,InChIKey,Definition,ParentIDs,TreeNumbers,ParentTreeNumbers,MESHSynonyms,CTDCuratedSynonyms,CTD_norm_name
0,0,(0.017ferrocene)amylose,MESH:C089250,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D000075163|MESH:D000688|MESH:D005296,D01.490.200/C089250|D02.691.657/C089250|D09.30...,D01.490.200|D02.691.657|D09.301.915.361|D09.69...,(0.017 ferrocene)amylose,NaN,0 017ferrocene amylose
1,1,001-C8-NBD,MESH:C114385,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842|MESH:D010069,D03.383.129.462.580/C114385|D12.644.456/C114385,D03.383.129.462.580|D12.644.456,001 C8 NBD|H-MeTyr-Arg-MeArg-D-Leu-NH(CH2)8NH-...,NaN,001 c8 nbd
2,2,001-C8 oligopeptide,MESH:C114386,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D009842,D12.644.456/C114386,D12.644.456,001 C8 oligopeptide|H-MeTyr-Arg-MeArg-D-Leu-NH...,NaN,001 c8 oligopeptide
3,3,"0231A , Streptomyces",MESH:C434150,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434150,D03.633.400,NaN,NaN,0231a streptomyces
4,4,"0231B, Streptomyces",MESH:C434149,NaN,NaN,NaN,NaN,NaN,NaN,MESH:D006576,D03.633.400/C434149,D03.633.400,NaN,NaN,0231b streptomyces


In [ ]:
import pandas as pd


# Load inputs

phase1_path = "./SIDER_LINCS_common_drug_ADR.csv"
ctd_path = "./CTD_chemicals_normalized.csv"

phase1_df = pd.read_csv(phase1_path)
ctd_df = pd.read_csv(ctd_path)


# Join key columns
# Phase-1: sider_norm_name
# CTD:     ctd_norm_name

phase1_drugs = phase1_df[["sider_norm_name"]].drop_duplicates()

drug_to_ctd = phase1_drugs.merge(
    ctd_df[["ChemicalID", "ChemicalName", "ctd_norm_name"]],
    left_on="sider_norm_name",
    right_on="ctd_norm_name",
    how="inner"
)


# Final output format

drug_to_ctd = (
    drug_to_ctd
    .rename(columns={
        "sider_norm_name": "canonical_drug_name",
        "ChemicalID": "CTD_ChemicalID",
        "ChemicalName": "CTD_ChemicalName"
    })[
        ["canonical_drug_name", "CTD_ChemicalID", "CTD_ChemicalName"]
    ]
)

drug_to_ctd.to_csv("./drug_to_ctd.csv", index=False)


# Drop Phase-1 drugs with zero CTD matches

valid_drugs = set(drug_to_ctd["canonical_drug_name"].unique())

# drugs_phase1.csv
drugs_phase1 = (
    phase1_df[["sider_norm_name"]]
    .drop_duplicates()
    .rename(columns={"sider_norm_name": "canonical_drug_name"})
    .query("canonical_drug_name in @valid_drugs")
)
drugs_phase1.to_csv("./drugs_phase1.csv", index=False)

# sider_labels_phase1.csv
sider_labels_phase1 = phase1_df.query(
    "sider_norm_name in @valid_drugs"
)
sider_labels_phase1.to_csv("./sider_labels_phase1.csv", index=False)


# Sanity checks

print("Unique Phase-1 drugs (before):", phase1_df["sider_norm_name"].nunique())
print("Phase-1 drugs after CTD join:", len(valid_drugs))
print("drug_to_ctd rows:", len(drug_to_ctd))


Unique Phase-1 drugs (before): 698
Phase-1 drugs after CTD join: 666
drug_to_ctd rows: 700


## ctd chem genes

In [115]:

df = pd.read_csv(
    "./CTD_chem_gene_ixns.tsv",
    sep="\t",
    engine="python",
    comment= '#',
)

print(len(df))
print(df.columns.tolist())


3019049
['10074-G5', 'C534883', 'Unnamed: 2', 'AR', '367', 'protein', 'Homo sapiens', '9606', '10074-G5 affects the reaction [MYC protein results in increased expression of AR protein]', 'affects^reaction|increases^expression', '32184358']


In [129]:
df = pd.read_csv("./CTD_chem_gene_ixns_clean.csv", header=None)
print(df.shape[1])
print(df.iloc[0].tolist())


12
[nan, '10074-G5', 'C534883', 'Unnamed: 2', 'AR', 367, 'protein', 'Homo sapiens', 9606.0, '10074-G5 affects the reaction [MYC protein results in increased expression of AR protein]', 'affects^reaction|increases^expression', '32184358']


In [130]:
df = df.iloc[:, 1:]

print(df.shape)        # must be (rows, 11)
print(df.iloc[0].tolist())

(3019050, 11)
['10074-G5', 'C534883', 'Unnamed: 2', 'AR', 367, 'protein', 'Homo sapiens', 9606.0, '10074-G5 affects the reaction [MYC protein results in increased expression of AR protein]', 'affects^reaction|increases^expression', '32184358']


In [131]:
output_csv = "./CTD_chem_gene_ixns_with_headers.csv"


df.columns = [
    "ChemicalName",
    "ChemicalID",
    "CasRN",
    "GeneSymbol",
    "GeneID",
    "GeneForms",
    "Organism",
    "OrganismID",
    "Interaction",
    "InteractionActions",
    "PubMedIDs"
]

df = pd.read_csv(
    input_csv,
    header=None
)


df.to_csv(output_csv, index=False)


In [134]:
import pandas as pd

in_path = "./CTD_chem_gene_ixns_with_headers.csv"   # the wrong one you just loaded
out_path = "./CTD_chem_gene_ixns_fixed.csv"         # corrected output

# Read without trusting existing headers
df = pd.read_csv(in_path, header=None)

# If first column is phantom (mostly empty / 0,1,2...), drop it
# This is safe because CTD schema starts with ChemicalName, not an empty index-like column.
df = df.iloc[:, 1:]  # drop column 0

# Now enforce the correct 11-column schema
df.columns = [
    "ChemicalName",
    "ChemicalID",
    "CasRN",
    "GeneSymbol",
    "GeneID",
    "GeneForms",
    "Organism",
    "OrganismID",
    "Interaction",
    "InteractionActions",
    "PubMedIDs"
]

# Save fixed CSV
df.to_csv(out_path, index=False)

# Sanity checks
print("Shape:", df.shape)                 # must be (rows, 11)
print("Columns:", df.columns.tolist())
print(df.head())


Shape: (3019051, 11)
Columns: ['ChemicalName', 'ChemicalID', 'CasRN', 'GeneSymbol', 'GeneID', 'GeneForms', 'Organism', 'OrganismID', 'Interaction', 'InteractionActions', 'PubMedIDs']
  ChemicalName ChemicalID       CasRN GeneSymbol  GeneID GeneForms  \
0            1          2           3          4       5         6   
1     10074-G5    C534883  Unnamed: 2         AR     367   protein   
2     10074-G5    C534883         NaN         AR     367   protein   
3     10074-G5    C534883         NaN         AR     367   protein   
4     10074-G5    C534883         NaN         AR     367   protein   

       Organism  OrganismID  \
0             7         8.0   
1  Homo sapiens      9606.0   
2  Homo sapiens      9606.0   
3  Homo sapiens      9606.0   
4  Homo sapiens      9606.0   

                                         Interaction  \
0                                                  9   
1  10074-G5 affects the reaction [MYC protein res...   
2  10074-G5 inhibits the reaction [EPHB2 

In [141]:
def normalize_name(s):
    if pd.isna(s):
        return s
    s = s.lower()
    s = re.sub(r"[^\w\s]", "", s)
    for salt in [
        "hydrochloride", "sodium", "potassium", "acetate",
        "sulfate", "phosphate", "nitrate", "tartrate"
    ]:
        s = s.replace(salt, "")
    return s.strip()

drug_to_ctd = pd.read_csv("./drug_to_ctd.csv")
ctd_cgi = pd.read_csv("./CTD_chem_gene_ixns_fixed.csv")

drug_to_ctd["ctd_norm_name"] = drug_to_ctd["CTD_ChemicalName"].apply(normalize_name)
ctd_cgi["ctd_norm_name"] = ctd_cgi["ChemicalName"].apply(normalize_name)

drug_gene = drug_to_ctd.merge(
    ctd_cgi[["ctd_norm_name", "GeneSymbol"]].dropna(),
    on="ctd_norm_name",
    how="inner"
)

drug_gene = drug_gene.rename(columns={"GeneSymbol": "gene_symbol"})[
    ["canonical_drug_name", "gene_symbol"]
].drop_duplicates()

drug_counts = drug_gene.groupby("canonical_drug_name")["gene_symbol"].nunique()

print("drugs:", drug_gene["canonical_drug_name"].nunique())
print("genes:", drug_gene["gene_symbol"].nunique())
print("min_genes_per_drug:", drug_counts.min())
print("max_genes_per_drug:", drug_counts.max())

drug_gene.to_csv("./drug_gene_edges.csv", index=False)


drugs: 638
genes: 29758
min_genes_per_drug: 1
max_genes_per_drug: 13280


In [140]:
print(ctd_cgi.columns.tolist())

['ChemicalName', 'ChemicalID', 'CasRN', 'GeneSymbol', 'GeneID', 'GeneForms', 'Organism', 'OrganismID', 'Interaction', 'InteractionActions', 'PubMedIDs']


In [ ]:
# find last column index that has any non-null value
last_valid_col = df.notna().any().to_numpy().nonzero()[0].max()

# keep columns up to that index
df = df.iloc[:, : last_valid_col + 1]

print(df.shape)
print(df.iloc[0].tolist())


(108440, 8)
['06-Paris-LA-66 protocol', 'C046983', '', 'Precursor Cell Lymphoblastic Leukemia-Lymphoma', 'MESH:D054198', 'therapeutic', '4519131', None]


## ctd chem diseases:

In [2]:
with open("./CTD_curated_chemicals_diseases.tsv", "r", encoding="utf-8") as f:
    for line in f:
        if not line.startswith("#"):
            print(line)
            break

06-Paris-LA-66 protocol	C046983		Precursor Cell Lymphoblastic Leukemia-Lymphoma	MESH:D054198	therapeutic	4519131



In [5]:
import pandas as pd

In [7]:

df = pd.read_csv(
    "./CTD_curated_chemicals_diseases.tsv",
    sep="\t",
    comment="#",
    dtype=str,
    low_memory=False
)

In [8]:
print(df.columns.tolist())


['06-Paris-LA-66 protocol', 'C046983', 'Unnamed: 2', 'Precursor Cell Lymphoblastic Leukemia-Lymphoma', 'MESH:D054198', 'therapeutic', '4519131']


In [9]:
df.head()

,06-Paris-LA-66 protocol,C046983,Unnamed: 2,Precursor Cell Lymphoblastic Leukemia-Lymphoma,MESH:D054198,therapeutic,4519131
0,"10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone",C112297,NaN,Hyperkinesis,MESH:D006948,marker/mechanism,19098162
1,"10,10-bis(4-pyridinylmethyl)-9(10H)-anthracenone",C112297,NaN,Seizures,MESH:D012640,marker/mechanism,26348896
2,"10,11-dihydro-10-hydroxycarbamazepine",C039775,NaN,Epilepsy,MESH:D004827,therapeutic,17516704
3,"10,11-dihydroxy-N-n-propylnorapomorphine",C425777,NaN,Hyperkinesis,MESH:D006948,marker/mechanism,15765258
4,10-hydroxycamptothecin,C028098,67656-30-8,Huntington Disease,MESH:D006816,therapeutic,21909362


In [14]:
df.columns = [
    "ChemicalName",
    "ChemicalID",
    "CasRN",
    "DiseaseName",
    "DiseaseID",
    "DirectEvidence",
    "PubMedIDs",
]
# df.head(20)
# df.columns.tolist()


In [13]:
df.to_csv("CTD_curated_chemicals_diseases_clean.csv", index=False)

In [16]:
import re

def normalize_disease(x):
    x = x.lower()
    x = re.sub(r"[^a-z0-9\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["DiseaseName_norm"] = df["DiseaseName"].apply(normalize_disease)
df["DiseaseID"].str.startswith("MESH:").value_counts()
df["DirectEvidence"].value_counts()



DirectEvidence
marker/mechanism    69141
therapeutic         39298
Name: count, dtype: int64

## Map ADRs to CTD diseases

In [18]:
# Phase 1 drugs
drugs = pd.read_csv("drugs_phase1.csv", dtype=str)

# SIDER + LINCS common data
sider_lincs = pd.read_csv("SIDER_LINCS_common_drug_ADR.csv", dtype=str)

# CTD curated gene–disease
ctd = pd.read_csv(
    "CTD_curated_chemicals_diseases_clean.csv",
    dtype=str
)

In [19]:
phase1_drugs = set(drugs["canonical_drug_name"])

sider_lincs_p1 = sider_lincs[
    sider_lincs["drug_name"].isin(phase1_drugs)
].copy()


In [20]:
adr_phase1 = (
    sider_lincs_p1["adr_id"]
    .dropna()
    .unique()
)

adr_phase1 = pd.DataFrame(
    {"adr_id": adr_phase1}
)


In [32]:
medic = pd.read_csv(
    "./CTD_diseases.tsv",
    sep="\t",
    comment="#",
    header=None,
    dtype=str,
    engine="python"
)

# print(medic_raw.shape)
# print(medic_raw.head(9))
medic.columns = [
    "DiseaseName",          # 0
    "DiseaseID",            # 1
    "DefinitionRef",        # 2  (DOID / OMIM refs)
    "Definition",           # 3  (free text)
    "ParentIDs",            # 4
    "TreeNumbers",          # 5
    "ParentTreeNumbers",    # 6
    "Synonyms",             # 7
    "SlimMappings",         # 8
]

# medic.head(10)
medic.to_csv("./CTD_diseases.csv")


# HPO

In [ ]:
adr_raw = pd.read_csv(
    "./meddra_all_se.tsv",
    sep="\t",
    # compression="gzip",
    header=None,
    dtype=str
)

adr_raw.columns = [
    "stitch_id_flat",
    "stitch_id_stereo",
    "meddra_concept_id",
    "meddra_level",
    "meddra_pt_id",
    "adr_name"
]
# adr_raw.head()


,stitch_id_flat,stitch_id_stereo,meddra_concept_id,meddra_level,meddra_pt_id,adr_name
0,CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps
1,CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain
2,CID100000085,CID000010917,C0000737,LLT,C0000737,Abdominal pain
3,CID100000085,CID000010917,C0000737,PT,C0687713,Gastrointestinal pain
4,CID100000085,CID000010917,C0000737,PT,C0000737,Abdominal pain


In [79]:
adr_pt = adr_raw[adr_raw["meddra_level"] == "PT"].copy()
adr_map = adr_pt[["meddra_pt_id", "adr_name"]].drop_duplicates()
adr_map = adr_map.rename(columns={"meddra_pt_id": "adr_id"})
# adr_map.head(10)
adr_map.to_csv("./ADR_ID_name_map.csv")


In [84]:
adr_map.head()

,Unnamed: 0,adr_id,adr_name
0,1,C0000737,Abdominal pain
1,3,C0687713,Gastrointestinal pain
2,6,C0002418,Amblyopia
3,8,C0002871,Anaemia
4,10,C0232462,Decreased appetite


In [80]:
sider = pd.read_csv("SIDER_LINCS_common_drugs.csv", dtype=str)

sider = sider.merge(
    adr_map,
    on="adr_id",
    how="left"
)


In [85]:
sider.head()

,drug_id,drug_cid,drug_name,sider_norm_name,pert_id,pert_iname,adr_id,adr_name_x,Unnamed: 0,adr_name_y
0,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0002170,Alopecia,323,Alopecia
1,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0232462,Decreased appetite,10,Decreased appetite
2,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009450,Infection,28,Infection
3,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009806,Constipation,30,Constipation
4,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0011603,Dermatitis,47,Dermatitis


In [88]:
sider.to_csv("./sider_chemNames_adrNames.csv")

In [87]:
sider["adr_name_y"].notna().value_counts()


adr_name_y
True    137270
Name: count, dtype: int64

In [89]:
from collections import defaultdict
adr_map = pd.read_csv(
    "ADR_ID_name_map.csv",
    dtype=str
)

# expected columns: adr_id, adr_name
sider = sider.merge(
    adr_map,
    on="adr_id",
    how="left"
)
sider["adr_name_y"].notna().value_counts()

adr_name_y
True    137270
Name: count, dtype: int64

In [90]:
unique_adrs = (
    sider[["adr_id", "adr_name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

unique_adrs["adr_norm"] = unique_adrs["adr_name"].apply(normalize)
unique_adrs["adr_tokens"] = unique_adrs["adr_norm"].apply(lambda x: set(x.split()))


In [91]:
STOP_TOKENS = {
    "nos", "not", "otherwise", "specified",
    "abnormal", "increased", "decreased",
    "level", "levels", "disease", "disorder"
}

token_to_hpo = defaultdict(set)

with open("hp.obo", "r", encoding="utf-8") as f:
    current_id = None
    names = []

    for line in f:
        line = line.strip()

        if line == "[Term]":
            current_id = None
            names = []

        elif line.startswith("id: HP:"):
            current_id = line.replace("id: ", "").strip()

        elif line.startswith("name: "):
            names.append(line.replace("name: ", "").strip())

        elif line.startswith("synonym:"):
            parts = line.split('"')
            if len(parts) >= 2:
                names.append(parts[1])

        elif line == "" and current_id:
            for name in names:
                tokens = set(normalize(name).split()) - STOP_TOKENS
                for t in tokens:
                    token_to_hpo[t].add(current_id)


In [92]:
len(token_to_hpo)


12609

In [93]:
def map_adr_to_hpo(adr_tokens):
    adr_tokens = adr_tokens - STOP_TOKENS
    if not adr_tokens:
        return None

    candidate_hpos = set()
    for t in adr_tokens:
        candidate_hpos |= token_to_hpo.get(t, set())

    if not candidate_hpos:
        return None

    # choose one deterministically for now
    return next(iter(candidate_hpos))


In [94]:
unique_adrs["HPO_ID"] = unique_adrs["adr_tokens"].apply(map_adr_to_hpo)
unique_adrs["HPO_ID"].notna().value_counts()


HPO_ID
True     3396
False     263
Name: count, dtype: int64

In [95]:
sider = sider.merge(
    unique_adrs[["adr_id", "HPO_ID"]],
    on="adr_id",
    how="left"
)
sider.head()

,drug_id,drug_cid,drug_name,sider_norm_name,pert_id,pert_iname,adr_id,adr_name_x,Unnamed: 0_x,adr_name_y,Unnamed: 0_y,adr_name,HPO_ID
0,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0002170,Alopecia,323,Alopecia,323,Alopecia,HP:0007534
1,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0232462,Decreased appetite,10,Decreased appetite,10,Decreased appetite,HP:0004396
2,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009450,Infection,28,Infection,28,Infection,HP:0033141
3,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009806,Constipation,30,Constipation,30,Constipation,HP:0012450
4,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0011603,Dermatitis,47,Dermatitis,47,Dermatitis,HP:0001047


In [100]:
phen2gene = pd.read_csv(
    "phenotype_to_genes.txt",
    sep="\t",
    comment="#",
    dtype=str
)
phen2gene= phen2gene.rename(columns={"hpo_id":"HPO_ID"})
phen2gene.head()

,HPO_ID,hpo_name,ncbi_gene_id,gene_symbol,disease_id
0,HP:0025700,Anhydramnios,26281,FGF20,OMIM:615721
1,HP:0025700,Anhydramnios,2674,GFRA1,OMIM:619887
2,HP:0025700,Anhydramnios,79867,TCTN2,OMIM:613885
3,HP:0025700,Anhydramnios,6091,ROBO1,OMIM:620305
4,HP:0025700,Anhydramnios,8516,ITGA8,OMIM:191830


In [ ]:
adr_hpo_genes = (
    unique_adrs
    .merge(phen2gene, on="HPO_ID", how="inner")
)

In [103]:
final_adr_hpo_genes = adr_hpo_genes[
    [
        "adr_id",
        "adr_name",
        "HPO_ID",
        "hpo_name",
        "gene_symbol",
        "ncbi_gene_id"
    ]
].drop_duplicates()


In [106]:
final_adr_hpo_genes.groupby("adr_name")["gene_symbol"].nunique().sort_values(ascending=False).head(10)


adr_name
Vocal cord disorder                4087
Vocal cord paresis                 4087
Head discomfort                    3417
Head lag abnormal                  3417
Head titubation                    3417
Urogenital disorder                2737
Musculoskeletal disorder           1697
Psychiatric evaluation abnormal    1590
Painful respiration                1521
Respiration abnormal               1521
Name: gene_symbol, dtype: int64

In [107]:
final_adr_hpo_genes.head()

,adr_id,adr_name,HPO_ID,hpo_name,gene_symbol,ncbi_gene_id
0,C0002170,Alopecia,HP:0007534,Congenital posterior occipital alopecia,HRAS,3265
1,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT10,3858
2,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT1,3848
3,C0232462,Decreased appetite,HP:0004396,Poor appetite,IL18BP,10068
4,C0232462,Decreased appetite,HP:0004396,Poor appetite,TRANK1,9881


In [108]:
final_adr_hpo_genes.to_csv("./adr_hpo_gene_grounding.csv")

# adr->disease

In [113]:
sider.head()

,drug_id,drug_cid,drug_name,sider_norm_name,pert_id,pert_iname,adr_id
0,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0002170
1,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0232462
2,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009450
3,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0009806
4,CID100000143,100000143,leucovorin,leucovorin,BRD-A75919782,leucovorin,C0011603


In [ ]:
df= pd.read_csv("CTD_diseases.csv")
df = df.drop(columns=["Unnamed: 0"])
df.head()
print(df.columns)

Index(['Unnamed: 0', 'DiseaseName', 'DiseaseID', 'DefinitionRef', 'Definition',
       'ParentIDs', 'TreeNumbers', 'ParentTreeNumbers', 'Synonyms',
       'SlimMappings'],
      dtype='object')


In [117]:
# Clean CTD diseases
ctd = df.copy()  # df is CTD_diseases
ctd = ctd.drop(columns=["Unnamed: 0"])
sider= pd.read_csv("sider_lincs_comm_chemNames_adrNames.csv")
# Drop duplicate / junk columns
sider = sider.drop(columns=["adr_name_y", "Unnamed: adr_name"], errors="ignore")

# Rename adr_name_x → adr_name
sider = sider.rename(columns={"adr_name_x": "adr_name"})

# Optional sanity check
print(
    sider[["adr_id", "adr_name"]]
    .drop_duplicates()
    .head(10)
)


     adr_id            adr_name
0  C0002170            Alopecia
1  C0232462  Decreased appetite
2  C0009450           Infection
3  C0009806        Constipation
4  C0011603          Dermatitis
5  C0011991           Diarrhoea
6  C0015230                Rash
7  C0015376       Extravasation
8  C0015672             Fatigue
9  C0004093            Asthenia


In [119]:
sider.to_csv("sider_lincs_common_adrid_adrname.csv")

In [118]:


def normalize(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Build name → DiseaseID map
name_to_disease = {}

def add_alias(name, did):
    name = normalize(name)
    if name and name not in name_to_disease:
        name_to_disease[name] = did

for _, row in ctd.iterrows():
    add_alias(row["DiseaseName"], row["DiseaseID"])

    if pd.notna(row["Synonyms"]):
        for syn in row["Synonyms"].split("|"):
            add_alias(syn, row["DiseaseID"])

# Load unique ADRs from SIDER–LINCS common
# sider = pd.read_csv("SIDER_LINCS_common_drugs.csv", dtype=str)
adrs = sider[["adr_id", "adr_name"]].drop_duplicates()
adrs["adr_norm"] = adrs["adr_name"].apply(normalize)

# Map ADR → DiseaseID
adrs["CTD_DiseaseID"] = adrs["adr_norm"].map(name_to_disease)

# Coverage check
print(adrs["CTD_DiseaseID"].notna().value_counts())


CTD_DiseaseID
False    2365
True     1294
Name: count, dtype: int64


In [120]:
adrs[adrs["CTD_DiseaseID"].notna()].sample(10)


,adr_id,adr_name,adr_norm,CTD_DiseaseID
1881,C0002962,Angina pectoris,angina pectoris,MESH:D000787
7037,C0151889,Hyperreflexia,hyperreflexia,MESH:D012021
6117,C0032227,Pleural effusion,pleural effusion,MESH:D010996
32298,C0085580,Essential hypertension,essential hypertension,MESH:D000075222
19669,C0155503,Vertigo CNS origin,vertigo cns origin,MESH:D014717
73017,C0019322,Umbilical hernia,umbilical hernia,MESH:D006554
2107,C0018800,Cardiomegaly,cardiomegaly,MESH:D006332
96359,C0877792,Circadian rhythm sleep disorder,circadian rhythm sleep disorder,MESH:D020178
132,C0039231,Tachycardia,tachycardia,MESH:D013610
6015,C0027059,Myocarditis,myocarditis,MESH:D009205


In [122]:
adrs[adrs["CTD_DiseaseID"].notna()].to_csv("adr_disease_mapped.csv")

# adr->disease->gene

In [124]:
# Load CTD gene–disease (human only)
ctd_gd = pd.read_csv(
    "CTD_curated_genes_diseases_human_only.tsv", sep="\t",
    dtype=str
)

# Inspect columns once
print(ctd_gd.columns)


Index(['GeneSymbol', 'GeneID', 'DiseaseName', 'DiseaseID', 'DirectEvidence',
       'InferenceGeneSymbol', 'PubMedIDs'],
      dtype='object')


In [125]:
# Keep only ADRs that mapped to CTD diseases
adr_ctd = adrs[adrs["CTD_DiseaseID"].notna()].copy()

# Reduce CTD gene–disease to necessary columns
ctd_gd_small = ctd_gd[
    ["DiseaseID", "GeneSymbol", "GeneID"]
].drop_duplicates()

# Join: ADR → Disease → Gene
adr_ctd_genes = adr_ctd.merge(
    ctd_gd_small,
    left_on="CTD_DiseaseID",
    right_on="DiseaseID",
    how="inner"
)

# Clean output
adr_ctd_genes = adr_ctd_genes[
    ["adr_id", "adr_name", "CTD_DiseaseID", "GeneSymbol", "GeneID"]
].drop_duplicates()

# Add provenance
adr_ctd_genes["source"] = "CTD_disease"

# Save
adr_ctd_genes.to_csv("adr_ctd_disease_genes.csv", index=False)

print(adr_ctd_genes.head())
print("Unique ADRs with CTD genes:", adr_ctd_genes["adr_id"].nunique())
print("Total ADR–gene rows:", len(adr_ctd_genes))


     adr_id  adr_name CTD_DiseaseID GeneSymbol GeneID       source
0  C0002170  Alopecia  MESH:D000505      ABCC2   1244  CTD_disease
1  C0002170  Alopecia  MESH:D000505         AR    367  CTD_disease
2  C0002170  Alopecia  MESH:D000505       BRD4  23476  CTD_disease
3  C0002170  Alopecia  MESH:D000505        CRH   1392  CTD_disease
4  C0002170  Alopecia  MESH:D000505         HR  55806  CTD_disease
Unique ADRs with CTD genes: 793
Total ADR–gene rows: 19513


# adr--> gene

In [130]:
# Load both grounding tables
hpo = pd.read_csv("adr_hpo_gene_grounding.csv", dtype=str)
ctd = pd.read_csv("adr_ctd_disease_genes.csv", dtype=str)

# Standardize column names
hpo = hpo.rename(columns={
    "gene_symbol": "GeneSymbol",
    "ncbi_gene_id": "GeneID"
})

hpo.head()

,Unnamed: 0,adr_id,adr_name,HPO_ID,hpo_name,GeneSymbol,GeneID
0,0,C0002170,Alopecia,HP:0007534,Congenital posterior occipital alopecia,HRAS,3265
1,1,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT10,3858
2,2,C0232462,Decreased appetite,HP:0004396,Poor appetite,KRT1,3848
3,3,C0232462,Decreased appetite,HP:0004396,Poor appetite,IL18BP,10068
4,4,C0232462,Decreased appetite,HP:0004396,Poor appetite,TRANK1,9881


In [131]:

# Keep only needed columns
hpo_small = hpo[["adr_id", "adr_name", "GeneSymbol", "GeneID"]].drop_duplicates()
hpo_small["CTD_DiseaseID"] = pd.NA
hpo_small["source"] = "HPO_phenotype"

ctd_small = ctd[["adr_id", "adr_name", "CTD_DiseaseID", "GeneSymbol", "GeneID", "source"]].drop_duplicates()

# Union
adr_gene_union = pd.concat(
    [
        hpo_small[ctd_small.columns],  # align schema
        ctd_small
    ],
    ignore_index=True
).drop_duplicates()

# Save final artifact
adr_gene_union.to_csv("adr_gene_grounding_union.csv", index=False)

print("Final ADR–gene grounding")
print("Unique ADRs:", adr_gene_union["adr_id"].nunique())
print("Unique genes:", adr_gene_union["GeneSymbol"].nunique())
print("Total rows:", len(adr_gene_union))


Final ADR–gene grounding
Unique ADRs: 2425
Unique genes: 8543
Total rows: 229271


In [132]:
sider = pd.read_csv("sider_lincs_comm_chemNames_adrNames.csv", dtype=str)

# Drop extra columns (safe because adr_name_x == adr_name_y everywhere)
sider = sider.drop(columns=["adr_name_y", "Unnamed: 0", "Unnamed: 0.1"], errors="ignore")

# Rename to a single canonical column name
sider = sider.rename(columns={"adr_name_x": "adr_name"})

sider.to_csv("sider_lincs_common_clean_FINAL.csv", index=False)
print(sider.columns)
print(sider.shape)


Index(['drug_id', 'drug_cid', 'drug_name', 'sider_norm_name', 'pert_id',
       'pert_iname', 'adr_id', 'adr_name'],
      dtype='object')
(137270, 8)


In [133]:
hpo = pd.read_csv("adr_hpo_gene_grounding.csv", dtype=str)
hpo = hpo.drop(columns=["Unnamed: 0"], errors="ignore")
hpo.to_csv("adr_hpo_gene_grounding_clean.csv", index=False)
print(hpo.columns)
print(hpo.shape)


Index(['adr_id', 'adr_name', 'HPO_ID', 'hpo_name', 'gene_symbol',
       'ncbi_gene_id'],
      dtype='object')
(209758, 6)
